# Todas las comparativas

Ejecuta todas las celdas para regenerar las cinco evidencias. El código está en este notebook y no usa los archivos de `test/`.


In [ ]:
RUN_MODE = "quick"  # "quick" mide 1K, 10K y 100K. "full" añade 1M y 10M.
SEED = 42
DATASET_FILE = ""  # Ruta a 2019-Oct.csv. Vacío: muestra sintética solo para validar.

print("Modo:", RUN_MODE)
print("Semilla:", SEED)
print("CSV:", DATASET_FILE or "sintético, solo validación")


In [ ]:
import platform
import time
import tracemalloc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "01_filtrado_probabilistico.ipynb").exists():
    ROOT = ROOT / "pruebas_algoritmos"

def sizes(mode):
    return [1_000, 10_000, 100_000] if mode == "quick" else [1_000, 10_000, 100_000, 1_000_000, 10_000_000]

def repetitions(n, mode):
    if mode == "quick":
        return 7
    return 3 if n >= 1_000_000 else 100

def prepare_output(folder):
    results, figures = folder / "resultados", folder / "figuras"
    results.mkdir(parents=True, exist_ok=True)
    figures.mkdir(parents=True, exist_ok=True)
    return results, figures

def resolve_data(data):
    if not data:
        return None
    candidate = Path(data)
    options = [candidate, Path.cwd() / candidate, ROOT / candidate, ROOT.parent / candidate]
    found = next((option for option in options if option.is_file()), None)
    if found is None:
        raise FileNotFoundError(f"No existe el CSV indicado: {data}")
    return found

def load_events(data, n, seed):
    found = resolve_data(data)
    if found:
        cols = ["event_time", "event_type", "product_id", "price", "user_id", "user_session", "brand", "category_code"]
        frame = pd.read_csv(found, nrows=n, usecols=lambda column: column in cols)
        return frame, "REES46 CSV"
    rng = np.random.default_rng(seed)
    products = max(300, min(n // 8, 20_000))
    users = max(200, min(n // 5, 50_000))
    frame = pd.DataFrame({
        "event_time": pd.date_range("2019-10-01", periods=n, freq="s"),
        "event_type": rng.choice(["view", "cart", "purchase"], size=n, p=[0.83, 0.12, 0.05]),
        "product_id": rng.integers(1_000_000, 1_000_000 + products, size=n),
        "price": np.round(rng.lognormal(4.2, 0.75, n), 2),
        "user_id": rng.integers(1, users, size=n),
        "user_session": rng.integers(1, max(20, n // 4), size=n).astype(str),
        "brand": rng.choice(["samsung", "apple", "sony", "unknown"], size=n),
        "category_code": rng.choice(["electronics.phone", "appliances.kitchen", "computers"], size=n),
    })
    return frame, "SINTÉTICO (solo validación; no usar en el informe)"

def measure(action, reps):
    timings = []
    for _ in range(reps):
        start = time.perf_counter()
        action()
        timings.append((time.perf_counter() - start) * 1_000)
    tracemalloc.start()
    action()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return float(np.mean(timings)), float(np.std(timings)), peak / 1024 ** 2

def record(rows, **fields):
    rows.append({
        "experimento": fields["experiment"], "algoritmo": fields["algorithm"], "n": fields["n"],
        "metrica": fields["metric"], "valor": fields["value"], "tiempo_promedio_ms": fields["time_ms"],
        "desviacion_ms": fields["std_ms"], "memoria_pico_mb": fields["memory_mb"],
        "repeticiones": fields["reps"], "semilla": fields["seed"],
        "plataforma": platform.platform(), "fuente": fields["source"],
    })

def save(rows, name, output):
    results, _ = prepare_output(output)
    path = results / f"{name}.csv"
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"Tabla: {path}")
    return path

def line_chart(frame, title, filename, output, value="tiempo_promedio_ms", log=True):
    _, figures = prepare_output(output)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for algorithm, group in frame.groupby("algoritmo"):
        group = group.sort_values("n")
        if log:
            group = group[group[value] > 0]
        if group.empty:
            continue
        ax.plot(group["n"], group[value], marker="o", label=algorithm)
    ax.set(title=title, xlabel="Elementos", ylabel=value.replace("_", " "))
    if log:
        ax.set_xscale("log")
        ax.set_yscale("log")
    ax.grid(alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    path = figures / filename
    fig.savefig(path, dpi=160)
    plt.show()
    plt.close(fig)
    print(f"Figura: {path}")
    return path

import hashlib
import heapq
import math
import random
import importlib.util
import collections
from collections import defaultdict
from itertools import combinations
import networkx as nx
def stable_hash(value: object, salt: int = 0) -> int:
    return int.from_bytes(hashlib.blake2b(f"{salt}|{value}".encode(), digest_size=8).digest(), "little")


class BloomFilter:
    def __init__(self, capacity: int, error_rate: float = 0.01):
        self.bits_count = max(8, int(-capacity * math.log(error_rate) / math.log(2) ** 2))
        self.hashes = max(1, round(self.bits_count / max(capacity, 1) * math.log(2)))
        self.bits = bytearray((self.bits_count + 7) // 8)

    def _locations(self, value: object):
        for salt in range(self.hashes):
            yield stable_hash(value, salt) % self.bits_count

    def add(self, value: object) -> None:
        for pos in self._locations(value):
            self.bits[pos // 8] |= 1 << (pos % 8)

    def contains(self, value: object) -> bool:
        return all(self.bits[pos // 8] & (1 << (pos % 8)) for pos in self._locations(value))


class CountMinSketch:
    def __init__(self, width: int = 2048, depth: int = 5):
        self.width, self.depth = width, depth
        self.table = np.zeros((depth, width), dtype=np.uint32)

    def add(self, value: object, count: int = 1) -> None:
        for row in range(self.depth):
            self.table[row, stable_hash(value, row) % self.width] += count

    def estimate(self, value: object) -> int:
        return int(min(self.table[row, stable_hash(value, row) % self.width] for row in range(self.depth)))


class MinHashLSH:
    def __init__(self, permutations: int = 32, bands: int = 8):
        self.permutations, self.bands = permutations, bands
        self.rows = permutations // bands
        self.signatures: dict[int, tuple[int, ...]] = {}
        self.buckets: list[dict[tuple[int, ...], set[int]]] = [defaultdict(set) for _ in range(bands)]

    def signature(self, values: set[int]) -> tuple[int, ...]:
        return tuple(min((stable_hash(v, seed) for v in values), default=0) for seed in range(self.permutations))

    def add(self, key: int, values: set[int]) -> None:
        sig = self.signature(values)
        self.signatures[key] = sig
        for band in range(self.bands):
            start = band * self.rows
            self.buckets[band][sig[start:start + self.rows]].add(key)

    def candidates(self, key: int) -> set[int]:
        sig = self.signatures[key]
        hits: set[int] = set()
        for band in range(self.bands):
            start = band * self.rows
            hits.update(self.buckets[band][sig[start:start + self.rows]])
        hits.discard(key)
        return hits


def simhash(values: set[int], bits: int = 32) -> int:
    accumulator = [0] * bits
    for value in values:
        hashed = stable_hash(value)
        for bit in range(bits):
            accumulator[bit] += 1 if hashed & (1 << bit) else -1
    return sum((1 << bit) for bit, score in enumerate(accumulator) if score >= 0)


def hamming(a: int, b: int) -> int:
    return bin(a ^ b).count("1")


class IVF:
    """Índice vectorial por clusters, deliberadamente simple para comparar trade-offs."""
    def __init__(self, clusters: int = 32, iterations: int = 8, seed: int = 42):
        self.clusters, self.iterations, self.seed = clusters, iterations, seed

    def fit(self, vectors: np.ndarray, ids: np.ndarray) -> None:
        self.vectors, self.ids = vectors.astype(np.float32), ids
        rng = np.random.default_rng(self.seed)
        centers = self.vectors[rng.choice(len(vectors), min(self.clusters, len(vectors)), replace=False)].copy()
        for _ in range(self.iterations):
            labels = ((self.vectors[:, None] - centers[None, :]) ** 2).sum(axis=2).argmin(axis=1)
            for cluster in range(len(centers)):
                group = self.vectors[labels == cluster]
                if len(group):
                    centers[cluster] = group.mean(axis=0)
        self.centers, self.labels = centers, labels

    def query(self, vector: np.ndarray, k: int, probes: int = 4) -> list[int]:
        chosen = np.argsort(((self.centers - vector) ** 2).sum(axis=1))[:probes]
        positions = np.flatnonzero(np.isin(self.labels, chosen))
        distance = ((self.vectors[positions] - vector) ** 2).sum(axis=1)
        return self.ids[positions[np.argsort(distance)[:k]]].tolist()


def exact_knn(vectors: np.ndarray, ids: np.ndarray, query: np.ndarray, k: int) -> list[int]:
    distance = ((vectors - query) ** 2).sum(axis=1)
    return ids[np.argsort(distance)[:k]].tolist()


class KDTree:
    def __init__(self, vectors: np.ndarray, ids: np.ndarray):
        self.vectors, self.ids = vectors, ids
        self.root = self._build(np.arange(len(ids)))

    def _build(self, positions: np.ndarray):
        if not len(positions):
            return None
        axis = int(np.var(self.vectors[positions], axis=0).argmax())
        ordered = positions[np.argsort(self.vectors[positions, axis])]
        middle = len(ordered) // 2
        return axis, ordered[middle], self._build(ordered[:middle]), self._build(ordered[middle + 1:])

    def query(self, vector: np.ndarray, k: int) -> list[int]:
        heap: list[tuple[float, int]] = []
        def visit(node):
            if node is None:
                return
            axis, pos, left, right = node
            squared = float(((self.vectors[pos] - vector) ** 2).sum())
            heapq.heappush(heap, (-squared, pos))
            if len(heap) > k:
                heapq.heappop(heap)
            near, far = (left, right) if vector[axis] < self.vectors[pos, axis] else (right, left)
            visit(near)
            if len(heap) < k or (vector[axis] - self.vectors[pos, axis]) ** 2 < -heap[0][0]:
                visit(far)
        visit(self.root)
        return self.ids[[pos for _, pos in sorted(heap, reverse=True)]].tolist()

def run_probabilisticos(mode="quick", data=None, seed=42, out=None) -> pd.DataFrame:
    out = ROOT
    rows = []
    max_n = max(sizes(mode))
    df, source = load_events(data, max_n, seed)
    values = df["user_id"].astype(int).to_numpy()
    for n in sizes(mode):
        sample, reps = values[:n], repetitions(n, mode)
        inserted = set(sample[::10])
        negatives = [int(v) for v in sample if v not in inserted][:10_000]
        bloom = BloomFilter(len(inserted))
        bloom_time, bloom_std, bloom_mem = measure(lambda: [bloom.add(int(v)) for v in inserted], reps)
        for value in inserted:
            bloom.add(value)
        fp = sum(bloom.contains(v) for v in negatives) / max(1, len(negatives))
        record(rows, experiment="Bloom vs set", algorithm="Bloom Filter", n=n, metric="falsos_positivos",
               value=fp, time_ms=bloom_time, std_ms=bloom_std, memory_mb=bloom_mem, reps=reps, seed=seed, source=source)
        exact_time, exact_std, exact_mem = measure(lambda: set(inserted), reps)
        record(rows, experiment="Bloom vs set", algorithm="set exacto", n=n, metric="falsos_positivos",
               value=0, time_ms=exact_time, std_ms=exact_std, memory_mb=exact_mem, reps=reps, seed=seed, source=source)

        cms = CountMinSketch()
        cms_time, cms_std, cms_mem = measure(lambda: [cms.add(int(v)) for v in sample], reps)
        for value in sample:
            cms.add(int(value))
        exact = collections.Counter(map(int, sample))
        mae = sum(abs(cms.estimate(key) - count) for key, count in exact.items()) / max(1, len(exact))
        record(rows, experiment="CMS vs Counter", algorithm="Count-Min Sketch", n=n, metric="mae_frecuencia",
               value=mae, time_ms=cms_time, std_ms=cms_std, memory_mb=cms_mem, reps=reps, seed=seed, source=source)
        counter_time, counter_std, counter_mem = measure(lambda: collections.Counter(map(int, sample)), reps)
        record(rows, experiment="CMS vs Counter", algorithm="Counter exacto", n=n, metric="mae_frecuencia",
               value=0, time_ms=counter_time, std_ms=counter_std, memory_mb=counter_mem, reps=reps, seed=seed, source=source)
    path = save(rows, "probabilisticos", out)
    result = pd.read_csv(path)
    line_chart(result[result["experimento"] == "Bloom vs set"], "Bloom Filter vs set: tiempo", "bloom_tiempo.png", out)
    line_chart(result[result["experimento"] == "Bloom vs set"], "Bloom Filter: falsos positivos", "bloom_calidad.png", out, value="valor", log=False)
    line_chart(result[result["experimento"] == "CMS vs Counter"], "Count-Min vs Counter: tiempo", "cms_tiempo.png", out)
    line_chart(result[result["experimento"] == "CMS vs Counter"], "Count-Min: error de frecuencia", "cms_calidad.png", out, value="valor", log=False)
    return result

def relevant_sets(frame, maximum=250):
    groups = defaultdict(set)
    for product, user in zip(frame["product_id"], frame["user_id"]):
        groups[int(product)].add(int(user))
    return dict(sorted(groups.items(), key=lambda item: len(item[1]), reverse=True)[:maximum])


def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 0.0


def run_similitud(mode="quick", data=None, seed=42, out=None) -> pd.DataFrame:
    out = ROOT
    rows = []
    data, source = load_events(data, max(sizes(mode)), seed)
    for n in sizes(mode):
        sets, reps = relevant_sets(data.iloc[:n]), repetitions(n, mode)
        keys = list(sets)
        if len(keys) < 2:
            continue
        query = keys[0]
        truth = {key for key in keys[1:] if jaccard(sets[query], sets[key]) >= 0.03}
        lsh = MinHashLSH()
        time_lsh, std_lsh, mem_lsh = measure(lambda: [lsh.add(key, values) for key, values in sets.items()], reps)
        for key, values in sets.items():
            lsh.add(key, values)
        candidates = lsh.candidates(query)
        recall = len(candidates & truth) / len(truth) if truth else 1.0
        record(rows, experiment="conjuntos", algorithm="MinHash+LSH", n=n, metric="recall", value=recall,
               time_ms=time_lsh, std_ms=std_lsh, memory_mb=mem_lsh, reps=reps, seed=seed, source=source)
        signatures = {key: simhash(values) for key, values in sets.items()}
        time_sh, std_sh, mem_sh = measure(lambda: {key: simhash(values) for key, values in sets.items()}, reps)
        limit = 10
        sh_candidates = {key for key in keys[1:] if hamming(signatures[query], signatures[key]) <= limit}
        sh_recall = len(sh_candidates & truth) / len(truth) if truth else 1.0
        record(rows, experiment="conjuntos", algorithm="SimHash", n=n, metric="recall", value=sh_recall,
               time_ms=time_sh, std_ms=std_sh, memory_mb=mem_sh, reps=reps, seed=seed, source=source)
        time_exact, std_exact, mem_exact = measure(
            lambda: [jaccard(sets[query], sets[key]) for key in keys[1:]], reps)
        record(rows, experiment="conjuntos", algorithm="Jaccard exacto", n=n, metric="recall", value=1.0,
               time_ms=time_exact, std_ms=std_exact, memory_mb=mem_exact, reps=reps, seed=seed, source=source)
    path = save(rows, "similitud", out)
    result = pd.read_csv(path)
    line_chart(result, "Similitud de conjuntos: tiempo de construcción", "similitud_tiempo.png", out)
    line_chart(result, "Similitud de conjuntos: recall", "similitud_recall.png", out, value="valor", log=False)
    return result

def vectors(frame):
    catalog = frame.groupby("product_id").agg(price=("price", "median"), popularity=("user_id", "size")).head(20_000)
    values = catalog.to_numpy(dtype=np.float32)
    values = (values - values.mean(axis=0)) / np.maximum(values.std(axis=0), 1e-8)
    return values, catalog.index.to_numpy(dtype=np.int64)


def run_ann(mode="quick", data=None, seed=42, out=None) -> pd.DataFrame:
    out = ROOT
    rows = []
    events, source = load_events(data, max(sizes(mode)), seed)
    all_vectors, all_ids = vectors(events)
    if len(all_ids) < 10:
        raise RuntimeError("No hay suficientes productos únicos para ANN.")
    for target in sizes(mode):
        limit = min(len(all_ids), max(10, target // 10))
        vec, ids = all_vectors[:limit], all_ids[:limit]
        query, k, reps = vec[0], min(10, len(ids)), repetitions(target, mode)
        truth = exact_knn(vec, ids, query, k)
        exact_time, exact_std, exact_mem = measure(lambda: exact_knn(vec, ids, query, k), reps)
        record(rows, experiment="ANN", algorithm="KNN exacto", n=len(ids), metric="recall_at_k", value=1,
               time_ms=exact_time, std_ms=exact_std, memory_mb=exact_mem, reps=reps, seed=seed, source=source)
        ivf = IVF(clusters=min(32, len(ids)), seed=seed)
        build_time, build_std, build_mem = measure(lambda: ivf.fit(vec, ids), reps)
        ivf.fit(vec, ids)
        candidates = ivf.query(query, k)
        recall = len(set(truth) & set(candidates)) / k
        record(rows, experiment="ANN", algorithm="IVF", n=len(ids), metric="recall_at_k", value=recall,
               time_ms=build_time, std_ms=build_std, memory_mb=build_mem, reps=reps, seed=seed, source=source)
        if importlib.util.find_spec("hnswlib"):
            import hnswlib
            def build_hnsw():
                index = hnswlib.Index(space="l2", dim=vec.shape[1])
                index.init_index(max_elements=len(ids), ef_construction=100, M=16, random_seed=seed)
                index.add_items(vec, ids)
                index.set_ef(50)
                return index
            h_time, h_std, h_mem = measure(build_hnsw, reps)
            index = build_hnsw()
            labels, _ = index.knn_query(query, k=k)
            h_recall = len(set(truth) & set(labels[0])) / k
            record(rows, experiment="ANN", algorithm="HNSW", n=len(ids), metric="recall_at_k", value=h_recall,
                   time_ms=h_time, std_ms=h_std, memory_mb=h_mem, reps=reps, seed=seed, source=source)
        else:
            print("HNSW omitido: instala hnswlib para obtener la comparación completa.")
    path = save(rows, "ann", out)
    result = pd.read_csv(path)
    line_chart(result, "ANN: tiempo", "ann_tiempo.png", out)
    line_chart(result, "ANN: recall@K", "ann_recall.png", out, value="valor", log=False)
    return result

def graph_from(frame):
    graph = nx.Graph()
    buys = frame[frame["event_type"] == "purchase"]
    for _, group in buys.groupby("user_session"):
        products = list(map(int, group["product_id"].unique()))[:20]
        for a, b in combinations(products, 2):
            graph.add_edge(a, b, weight=graph.get_edge_data(a, b, {}).get("weight", 0) + 1)
    return graph


def pagerank(graph, damping=0.85, iterations=30):
    """Implementación sin SciPy, equivalente a la usada por el notebook v2."""
    nodes = list(graph)
    if not nodes:
        return {}
    scores = {node: 1 / len(nodes) for node in nodes}
    for _ in range(iterations):
        updated = {node: (1 - damping) / len(nodes) for node in nodes}
        for node in nodes:
            neighbors = list(graph[node].items())
            total = sum(data.get("weight", 1) for _, data in neighbors)
            if total == 0:
                for target in nodes:
                    updated[target] += damping * scores[node] / len(nodes)
            else:
                for target, data in neighbors:
                    updated[target] += damping * scores[node] * data.get("weight", 1) / total
        scores = updated
    return scores


def partition_agreement(first, second, seed):
    """Estabilidad: proporción de pares que conservan la co-pertenencia."""
    nodes = list(set().union(*first))
    if len(nodes) < 2:
        return 1.0
    labels = []
    for partition in (first, second):
        labels.append({node: index for index, group in enumerate(partition) for node in group})
    import random
    rng = random.Random(seed)
    pairs = [rng.sample(nodes, 2) for _ in range(min(2_000, len(nodes) * 2))]
    return sum((labels[0][a] == labels[0][b]) == (labels[1][a] == labels[1][b]) for a, b in pairs) / len(pairs)


def run_grafo(mode="quick", data=None, seed=42, out=None) -> pd.DataFrame:
    out = ROOT
    rows = []
    events, source = load_events(data, max(sizes(mode)), seed)
    for n in sizes(mode):
        graph, reps = graph_from(events.iloc[:n]), repetitions(n, mode)
        if graph.number_of_edges() == 0:
            continue
        degree = dict(graph.degree(weight="weight"))
        pr_time, pr_std, pr_mem = measure(lambda: pagerank(graph), reps)
        scores = pagerank(graph)
        top_pr = set(sorted(scores, key=scores.get, reverse=True)[:10])
        top_degree = set(sorted(degree, key=degree.get, reverse=True)[:10])
        record(rows, experiment="ranking", algorithm="PageRank", n=n, metric="solape_top10_grado",
               value=len(top_pr & top_degree) / 10, time_ms=pr_time, std_ms=pr_std, memory_mb=pr_mem,
               reps=reps, seed=seed, source=source)
        record(rows, experiment="ranking", algorithm="Grado ponderado", n=n, metric="solape_top10_grado",
               value=1, time_ms=0, std_ms=0, memory_mb=0, reps=reps, seed=seed, source=source)
        def louvain():
            return list(nx.community.louvain_communities(graph, weight="weight", seed=seed))
        l_time, l_std, l_mem = measure(louvain, reps)
        communities = louvain()
        mod = nx.community.modularity(graph, communities, weight="weight")
        record(rows, experiment="comunidades", algorithm="Louvain", n=n, metric="modularidad",
               value=mod, time_ms=l_time, std_ms=l_std, memory_mb=l_mem, reps=reps, seed=seed, source=source)
        record(rows, experiment="comunidades", algorithm="Louvain", n=n, metric="numero_comunidades",
               value=len(communities), time_ms=l_time, std_ms=l_std, memory_mb=l_mem, reps=reps, seed=seed, source=source)
        record(rows, experiment="comunidades", algorithm="Louvain", n=n, metric="tamano_mediano",
               value=float(sorted(map(len, communities))[len(communities) // 2]), time_ms=l_time, std_ms=l_std,
               memory_mb=l_mem, reps=reps, seed=seed, source=source)
        stable = partition_agreement(communities, list(nx.community.louvain_communities(graph, weight="weight", seed=seed + 1)), seed)
        record(rows, experiment="comunidades", algorithm="Louvain", n=n, metric="estabilidad_semilla",
               value=stable, time_ms=l_time, std_ms=l_std, memory_mb=l_mem, reps=reps, seed=seed, source=source)
        def lpa():
            return list(nx.community.asyn_lpa_communities(graph, weight="weight", seed=seed))
        p_time, p_std, p_mem = measure(lpa, reps)
        communities = lpa()
        mod = nx.community.modularity(graph, communities, weight="weight")
        record(rows, experiment="comunidades", algorithm="Label Propagation", n=n, metric="modularidad",
               value=mod, time_ms=p_time, std_ms=p_std, memory_mb=p_mem, reps=reps, seed=seed, source=source)
        record(rows, experiment="comunidades", algorithm="Label Propagation", n=n, metric="numero_comunidades",
               value=len(communities), time_ms=p_time, std_ms=p_std, memory_mb=p_mem, reps=reps, seed=seed, source=source)
        record(rows, experiment="comunidades", algorithm="Label Propagation", n=n, metric="tamano_mediano",
               value=float(sorted(map(len, communities))[len(communities) // 2]), time_ms=p_time, std_ms=p_std,
               memory_mb=p_mem, reps=reps, seed=seed, source=source)
        stable = partition_agreement(communities, list(nx.community.asyn_lpa_communities(graph, weight="weight", seed=seed + 1)), seed)
        record(rows, experiment="comunidades", algorithm="Label Propagation", n=n, metric="estabilidad_semilla",
               value=stable, time_ms=p_time, std_ms=p_std, memory_mb=p_mem, reps=reps, seed=seed, source=source)
    path = save(rows, "grafo", out)
    result = pd.read_csv(path)
    comunidades = result[(result["experimento"] == "comunidades") & (result["metrica"] == "modularidad")]
    line_chart(comunidades, "Comunidades: tiempo", "comunidades_tiempo.png", out)
    line_chart(comunidades, "Comunidades: modularidad", "comunidades_modularidad.png", out, value="valor", log=False)
    return result

def run_bases(mode="quick", data=None, seed=42, out=None) -> pd.DataFrame:
    out = ROOT
    rows = []
    events, source = load_events(data, max(sizes(mode)), seed)
    raw_ids = events["product_id"].astype(int).to_numpy()
    catalog = events.groupby("product_id").agg(price=("price", "median"), popularity=("user_id", "size"))
    raw_vectors = catalog.to_numpy(dtype=float)
    raw_vectors = (raw_vectors - raw_vectors.mean(axis=0)) / np.maximum(raw_vectors.std(axis=0), 1e-8)
    ids = catalog.index.to_numpy(dtype=np.int64)
    for n in sizes(mode):
        sample, reps = raw_ids[:n], repetitions(n, mode)
        target = -1
        linear_time, linear_std, linear_mem = measure(lambda: target in sample, reps)
        record(rows, experiment="lookup", algorithm="Búsqueda lineal", n=n, metric="encontrado", value=0,
               time_ms=linear_time, std_ms=linear_std, memory_mb=linear_mem, reps=reps, seed=seed, source=source)
        lookup = set(map(int, sample))
        hash_time, hash_std, hash_mem = measure(lambda: target in lookup, reps)
        record(rows, experiment="lookup", algorithm="Hash set", n=n, metric="encontrado", value=0,
               time_ms=hash_time, std_ms=hash_std, memory_mb=hash_mem, reps=reps, seed=seed, source=source)

        limit = min(len(ids), max(10, n // 10))
        vec, vec_ids, query = raw_vectors[:limit], ids[:limit], raw_vectors[0]
        tree_time, tree_std, tree_mem = measure(lambda: KDTree(vec, vec_ids).query(query, min(5, limit)), reps)
        exact_time, exact_std, exact_mem = measure(lambda: exact_knn(vec, vec_ids, query, min(5, limit)), reps)
        record(rows, experiment="knn", algorithm="KD-Tree", n=limit, metric="exactitud", value=1,
               time_ms=tree_time, std_ms=tree_std, memory_mb=tree_mem, reps=reps, seed=seed, source=source)
        record(rows, experiment="knn", algorithm="Fuerza bruta", n=limit, metric="exactitud", value=1,
               time_ms=exact_time, std_ms=exact_std, memory_mb=exact_mem, reps=reps, seed=seed, source=source)
    path = save(rows, "bases", out)
    result = pd.read_csv(path)
    line_chart(result, "Estructuras base: tiempo", "bases_tiempo.png", out)
    line_chart(result, "Estructuras base: memoria", "bases_memoria.png", out, value="memoria_pico_mb", log=False)
    return result

def mostrar(frame):
    try:
        display(frame)
    except NameError:
        print(frame.to_string(index=False))

for nombre, funcion in (
    ("probabilisticos", run_probabilisticos),
    ("similitud", run_similitud),
    ("ann", run_ann),
    ("grafo", run_grafo),
    ("bases", run_bases),
):
    print("\n==>", nombre)
    mostrar(funcion(mode=RUN_MODE, data=DATASET_FILE or None, seed=SEED))
